In [2]:
import numpy as np
import pandas as pd
from pathlib import Path


In [3]:
def readpdb_PRIME20(filename, length):
    pdb=[]
    columns = ['atom', 'anum', 'atom_name', 'aa_name', 'resid', 'x', 'y', 'z']

    with open(filename, 'r') as f:
        lines = f.readlines()

    last_resid=length
    chain_id=0
    
    for line in lines:
        if line.startswith("ATOM" or "HETATM"):          
#             components = line.split()
            resid      = int(line[22:26])
            
            if((last_resid==length) & (resid==1)):
                chain_id+=1
                
            last_resid = resid
            
            row = {
                'atom': line[1:6],
                'anum': int(line[7:11]),
                'atom_name': line[12:15].strip(),
                'aa_name': line[16:20].strip(), 
                'chain_code':line[21:22],
                'resid':resid,
                'x': float(line[30:38]),
                'y': float(line[39:46]),
                'z': float(line[47:54]),
                'chain_id':chain_id # 记录chain ID 号码
            }
                       
            pdb.append(row)
            
    df_pdb=pd.DataFrame(pdb)        
    return df_pdb

def dist(df_i, df_j, L):
    xi, yi, zi = df_i[['x', 'y', 'z']].iloc[0]
    xj, yj, zj = df_j[['x', 'y', 'z']].iloc[0]
    
    rxij = xi - xj
    ryij = yi - yj
    rzij = zi - zj
    
    rxij = rxij - round(rxij/L)*L
    ryij = ryij - round(ryij/L)*L
    rzij = rzij - round(rzij/L)*L
    
    d = np.sqrt(rxij**2 + ryij**2 + rzij**2)
    return d


def gyration_tensor(rx, ry, rz, L):
    """
    Calculate the gyration tensor for a group of atoms.
    atoms: DataFrame with 'x', 'y', 'z' columns for coordinates and 'mass' column for atom masses.
    """
    # Wrap based on the first atom
    rx0 = rx[0]; ry0 = ry[0]; rz0=rz[0]
    for i in range(1, len(rx)):
        rx0i= rx0 - rx[i]
        ry0i= ry0 - ry[i]
        rz0i= rz0 - rz[i]
        
        rx0i= rx0i - round(rx0i/L)*L
        ry0i= ry0i - round(ry0i/L)*L
        rz0i= rz0i - round(rz0i/L)*L      
        
        rx[i] = rx0 - rx0i
        ry[i] = ry0 - ry0i
        rz[i] = rz0 - rz0i
    
    # Calculate center of mass
    com_x = np.sum(rx)/len(rx)
    com_y = np.sum(ry)/len(ry)
    com_z = np.sum(rz)/len(rz)
    
    # Calculate the gyration tensor
    tensor = np.zeros((3, 3))
    for i in range(len(rx)):
        # Get the atom's coordinates relative to the center of mass
        dx = rx[i] - com_x
        dy = ry[i] - com_y
        dz = rz[i] - com_z

        tensor[0, 0] +=dx**2
        tensor[0, 1] +=dx * dy
        tensor[0, 2] +=dx * dz
        tensor[1, 1] +=dy**2
        tensor[1, 2] +=dy * dz
        tensor[2, 2] +=dz**2
        
    tensor[1, 0] = tensor[0, 1]  # Symmetry
    tensor[2, 0] = tensor[0, 2]  # Symmetry
    tensor[2, 1] = tensor[1, 2]  # Symmetry

    tensor /= len(rx) # This is gyration tensor
    eigenvalues, eigenvectors = np.linalg.eigh(tensor)
    return eigenvalues

def Calculate_Asphericity(pdbfile, length=14, chains=24, L=200.0):

    pep=readpdb_PRIME20(pdbfile, length) # pep contains pdb file info
    rx=[]; ry=[]; rz=[]
    A=[]; P=[] 
    for i in range(1, chains+1):

        # get all backbone atoms
        df_chain=pep[(pep['chain_id'] == i) & (pep['atom_name'].isin(['C', 'N', 'CA']))]
        rx = df_chain['x'].to_numpy()
        ry = df_chain['y'].to_numpy()
        rz = df_chain['z'].to_numpy()
        eigenvalues = gyration_tensor(rx, ry, rz, L)
        a = eigenvalues[0]
        b = eigenvalues[1]
        c = eigenvalues[2]
        ratio1 = eigenvalues[2]/(eigenvalues.sum())
        ratio2 = np.sqrt(c/(a + b))
        asphericity = 1 - (3 * (a * b + b * c + c * a)) / ((a+b+c)**2)
        prolateness = 0.5*(2*a-b-c)*(2*b-a-c)*(2*c-a-b)/((a**2 + b**2 + c**2 - a*b -a*c -b*c)**1.5)

        A.append(asphericity)
        P.append(prolateness)
        #     print(i, np.round(eigenvalues,4), "lz^2/(R^2)",ratio1, "lz/root(lx^2 + ly^2)", ratio2)
#         print(i, np.round(eigenvalues,4), asphericity)

    ave = np.mean(A)
    std = np.std(A)
    
    return ave, std

In [ ]:
# Load information for an example peptide with 1 run at 3 temperatures

pep = "A3C-0001"
seq = "MTCTAINVNTMVYN"
path = Path(f"{pep}-{seq}") # Run this notebook from the asphericity folder.
Ts = [316, 328, 340]
run = "run0034"
L = 200.0 # Cubic simulation box length in angstroms.
length = len(seq)
chains = 24


In [10]:
# Calculate the asphericity of individual peptide chains. 
rows = []
for T in Ts:
    pdb = readpdb_PRIME20(path / f"{T}K" / f"{run}.pdb", length)
    backbone = pdb.loc[pdb.atom_name.isin(["N", "CA", "C"])]
    for chain, atoms in backbone.groupby("chain_id"):
        rx = atoms.x.to_numpy(copy=True)
        ry = atoms.y.to_numpy(copy=True)
        rz = atoms.z.to_numpy(copy=True)
        a, b, c = gyration_tensor(rx, ry,rz, L)
        A = 1 - 3 * (a*b + b*c + c*a) / (a + b + c)**2
        rows.append([T, chain, A])

result = pd.DataFrame(rows, columns=["T", "chain_id", "asphericity"])
result.pivot(index="chain_id", columns="T", values="asphericity")


T,316,328,340
chain_id,,,
1,0.270868,0.595984,0.488692
2,0.285454,0.551431,0.221589
3,0.288832,0.649227,0.330999
4,0.514715,0.295888,0.443846
5,0.625560,0.603400,0.692324
6,0.532523,0.291680,0.615608
7,0.583694,0.631586,0.445844
8,0.616676,0.621610,0.631821
9,0.490112,0.334388,0.599215


In [ ]:
# Calculate the average asphericity of all peptide chains in a PDB file. 
rows = []
for T in Ts:
    A, A_std = Calculate_Asphericity(path / f"{T}K" / f"{run}.pdb", length, chains, L)
    print("Temperature = ", T ,"K , Asphericity = ", A, ", STD =", A_std)



Temperature =  316 K , Asphericity =  0.4595682344297584 , STD = 0.135975148389185
Temperature =  328 K , Asphericity =  0.5266465996508205 , STD = 0.18498860298533346
Temperature =  340 K , Asphericity =  0.5007351668650661 , STD = 0.14786279922481083
